# Flujo de Generación de Templates para Incorporación de un Nuevo Tipo de Log o Revisión de un Origen Conocido

```
Líneas reales de COV_AIS_LOG.log
     │
     ▼ [1] drain3 (TemplateMiner) — igual que hace 'drain_unified.py parsed3'
demo_ais_drain3_parsed_full.json
     │
     ▼ [2] extract_templates.process_single_origin()
demo_ais_template.json
     │
     ▼ [3] semantic_grouping.semantic_group_templates()
demo_ais_template_semantic_groups.json
     │
     ▼ [4] generate_regex_from_groups.process_groups_file()
demo_ais_template_semantic_groups_regex.json
```

**Requisitos**: `pip install drain3 scikit-learn numpy pandas`

In [ ]:
import sys, os, json

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
TEMPLATE_GEN_DIR = os.path.join(PROJECT_ROOT, 'template_generator')

for p in [PROJECT_ROOT, TEMPLATE_GEN_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

print(f"PROJECT_ROOT     = {PROJECT_ROOT}")
print(f"TEMPLATE_GEN_DIR  = {TEMPLATE_GEN_DIR}")

WORKDIR = os.path.join(PROJECT_ROOT, 'notebooks', '_orchestrator_demo_output')
DRAIN3_OUTPUT_FOLDER = os.path.join(WORKDIR, 'drain3_parsed_results')
TEMPLATES_FOLDER = os.path.join(WORKDIR, 'templates')
os.makedirs(DRAIN3_OUTPUT_FOLDER, exist_ok=True)
os.makedirs(TEMPLATES_FOLDER, exist_ok=True)

ORIGIN = "ais"
print(f"\nOrigen de ejemplo: '{ORIGIN}'  |  Salida en: {WORKDIR}")

---
## Dataset: líneas REALES de `logs/COV_AIS_LOG.log`

Se extrae solo el **mensaje** (después del bloque de timestamp/nivel/módulo repetido), que es lo que
realmente se le pasaría a Drain3 en el pipeline real (`parsed_logs/*.parquet` ya contendría el mensaje
aislado.

In [ ]:
# Mensajes reales (ya extraídos del contenido tras el segundo bloque
# '[WARN  ] <MODULO> - [CUSTOM]/[AIS]') de COV_AIS_LOG.log
ejemplo_logs = [
    "START DB Groovy",
    "END DB Groovy",
    "START DB Groovy",
    "END DB Groovy",
    "START DB Groovy",
    "END DB Groovy",
    "Query que lanza: insert into proddta.f00095_del  select * from proddta.f00095 where a1user='JDEBSSV' AND (A1UPMJ< proddta.TO_DATEJ(SYSDATE) OR (A1UPMJ= proddta.TO_DATEJ(SYSDATE) AND A1UPMT <replace(to_char(sysdate-1/24, 'HH24:MI:SS'),':','')))",
    "Query que lanza: delete from proddta.f00095 where a1user='JDEBSSV' AND (A1UPMJ< proddta.TO_DATEJ(SYSDATE) OR (A1UPMJ= proddta.TO_DATEJ(SYSDATE) AND A1UPMT < replace(to_char(sysdate - INTERVAL '5' MINUTE, 'HH24:MI:SS'),':','')))",
    "Query que lanza: update proddta.f4211 set sdurab=1 where sdkcoo='01009' and sddcto='SO' and sddoco = '26020146' and sdlnid = '1000'",
    "Query que lanza: update proddta.f4211 set sdurab=1 where sdkcoo='01009' and sddcto='SO' and sddoco = '26020174' and sdlnid = '1000'",
    "Parameter Not Found: Actividad",
    "Parameter Not Found: Offset",
    "Parameter Not Found: Next",
    "Parameter Not Found: VICadena3",
    "Parameter Not Found: VICadena4",
    "Orchestration Exception Detected, exception saved in file: /u01/oracle/jde_home/LOGS_ORCH/7306259076693799989",
    "Orchestration Exception Detected, exception saved in file: /u01/oracle/jde_home/LOGS_ORCH/1532915514702387495",
    "Orchestration Exception Detected, exception saved in file: /u01/oracle/jde_home/LOGS_ORCH/6900969972350204568",
    "Orchestration Exception Detected, exception written to DB: 1ad1a8d6-cc09-4ce3-b1fa-1f33e3b4c53c",
    "Orchestration Exception Detected, exception written to DB: 376c2f57-efe7-4424-9aba-d70f06e95ec6",
    "RequestMonitorManager - writeExceptiontoDB() - Exception Written To DB 1ad1a8d6-cc09-4ce3-b1fa-1f33e3b4c53c",
    "RequestMonitorManager - writeExceptiontoDB() - Exception Written To DB 376c2f57-efe7-4424-9aba-d70f06e95ec6",
    " *** Inicio Bloque de Groovy SR_CN_COV_SELECT_ARTICULOS_CLIENTE ***",
    " *** FIN Bloque de Groovy SR_CN_COV_SELECT_ARTICULOS_CLIENTE  ***",
]

print(f"{len(ejemplo_logs)} mensajes de ejemplo (reales, de COV_AIS_LOG.log):\n")
for m in ejemplo_logs:
    print(f"  - {m[:100]}")

In [ ]:
from drain.common.origin_discovery import load_config_for_origin
from drain.common.drain_profile import get_drain_profile_params
from drain.common.drain3_utils import build_template_miner_config
from drain.core import process_parsed_messages_with_drain3

REAL_ORIGIN = "ais"  # origen real del proyecto

# 1) Config específica del origen AIS (timestamp_patterns, ignore_patterns, multiline, etc.)
origin_config = load_config_for_origin(REAL_ORIGIN)
print(f"✓ Config cargada para origen '{REAL_ORIGIN}':")
print(f"  - multiline: {origin_config['multiline']}")
print(f"  - ignore_patterns: {len(origin_config['ignore_patterns'])}")

# 2) Parámetros de Drain: igual que hace 'parsed3' en drain_unified.py, se usa
#    el perfil DRAIN_CONFIG del origen si no se especifican por CLI.
profile_params = get_drain_profile_params(REAL_ORIGIN)
depth = profile_params['depth']
st = profile_params['st']
max_children = profile_params['max_children']
print(f"\n✓ Parámetros del perfil DRAIN_CONFIG para '{REAL_ORIGIN}': "
      f"depth={depth}, st={st}, max_children={max_children} (profile={profile_params['profile']})")

In [ ]:
# 3) Convertimos nuestros mensajes de ejemplo al formato 'records' que espera
#    process_parsed_messages_with_drain3 (lista de dicts con clave 'message'),
#    tal y como los produciría load_parsed_logs_for_origin() leyendo parsed_logs/.
records_with_message = [{"message": msg} for msg in ejemplo_logs]

# 4) Ejecutamos la MISMA función que usa el pipeline real en producción.
result = process_parsed_messages_with_drain3(
    records_with_message, depth, st, max_children, origin_config, collect_records=True
)

records_with_message = result['records']       # cada record ahora trae 'cluster_id' y 'template'
clusters_info = result['clusters_info']

import pandas as pd
pd.set_option('display.max_colwidth', 90)
df_drain = pd.DataFrame([
    {
        "mensaje_original": r.get("message"),
        "cluster_id": r.get("cluster_id"),
        "template": r.get("template"),
    }
    for r in records_with_message
])
df_drain

In [ ]:
print("LOG REAL → CLUSTER → TEMPLATE (usando el pipeline REAL del proyecto, origen 'ais'):\n")
for _, row in df_drain.iterrows():
    print(f"LOG:      {str(row['mensaje_original'])[:90]}")
    print(f"CLUSTER:  #{row['cluster_id']}")
    print(f"TEMPLATE: {row['template']}")
    print("-" * 70)

print(f"\nClusters finales descubiertos por Drain3 (origen '{REAL_ORIGIN}', ignore_patterns aplicados):\n")
for c in clusters_info:
    print(f"Cluster #{c['cluster_id']} (size={c['size']}): {c['template']}")

In [ ]:
# Construimos el fichero '<origin>_drain3_parsed_full.json' con la MISMA
# estructura que produce el pipeline real, para poder encadenar el paso
# REAL de extract_templates.process_single_origin() sin cambios.
import os
total_records = len(records_with_message)

drain3_full_json = {
    "origin": REAL_ORIGIN,
    "clusters": [
        {
            "cluster_id": c["cluster_id"],
            "template": c["template"],
            "size": c["size"],
            "percentage": c["size"] / total_records if total_records else 0,
        }
        for c in clusters_info
    ],
}

drain3_full_path = os.path.join(DRAIN3_OUTPUT_FOLDER, f"{ORIGIN}_drain3_parsed_full.json")
with open(drain3_full_path, "w", encoding="utf-8") as f:
    json.dump(drain3_full_json, f, ensure_ascii=False, indent=2)

print(f"💾 {drain3_full_path}")

---
## PASO [2/4] — Extract Templates

In [ ]:
from template_generator.extract_templates import process_single_origin

templates_path = process_single_origin(
    origin=ORIGIN,
    input_folder=DRAIN3_OUTPUT_FOLDER,
    output_folder=TEMPLATES_FOLDER,
)

with open(templates_path, encoding="utf-8") as f:
    templates_data = json.load(f)

print(f"\n✓ {len(templates_data['clusters'])} templates en: {templates_path}\n")
for c in templates_data["clusters"]:
    print(f"  #{c['cluster_id']} (size={c['size']}): {c['template']}")

---
## PASO [3/4] — Similitud Semántica & Agrupación Clusters

In [ ]:
from template_generator.semantic_grouping import semantic_group_templates

DISTANCE_THRESHOLD = 0.6

semantic_result = semantic_group_templates(templates_path, DISTANCE_THRESHOLD)

semantic_groups_path = os.path.join(TEMPLATES_FOLDER, f"{ORIGIN}_template_semantic_groups.json")
with open(semantic_groups_path, "w", encoding="utf-8") as f:
    json.dump(semantic_result, f, ensure_ascii=False, indent=2)

print(f"\n💾 {semantic_groups_path}\n")
print(f"Resultado: {len(templates_data['clusters'])} templates → {len(semantic_result['groups'])} grupos semánticos\n")

for g in semantic_result["groups"]:
    print(f"GRUPO #{g['group_id']}  (representante cluster #{g['representative_cluster_id']}, "
          f"{g['num_members']} miembro(s), total_size={g['total_size']})")
    print(f"  Representante: {g['representative_template']}")
    for m in g["member_cluster_ids"]:
        marca = "  ← representante" if m["cluster_id"] == g["representative_cluster_id"] else ""
        print(f"    - #{m['cluster_id']} (size={m['size']}): {m['template']}{marca}")
    print()

### Asignación manual de `event_type` / `severity`

Según el diseño real (`semantic_grouping.py` inicializa estos campos a `None` para que se editen
manualmente antes de generar las regex), aquí simulamos esa revisión manual sobre 2-3 grupos.

In [ ]:
# Heurística simple SOLO para la demo: asignar event_type/severity según
# palabras clave del representative_template (en el proyecto real esto
# lo haría un humano revisando el JSON directamente).
for g in semantic_result["groups"]:
    tpl = g["representative_template"].lower()
    if "exception" in tpl or "orchestration" in tpl:
        g["event_type"] = "orchestration_error"
        g["severity"] = "ERROR"
    elif "parameter not found" in tpl:
        g["event_type"] = "missing_parameter"
        g["severity"] = "WARN"
    elif "groovy" in tpl:
        g["event_type"] = "groovy_block"
        g["severity"] = "INFO"
    elif "query" in tpl or "select" in tpl or "update" in tpl or "insert" in tpl or "delete" in tpl:
        g["event_type"] = "db_query"
        g["severity"] = "INFO"

with open(semantic_groups_path, "w", encoding="utf-8") as f:
    json.dump(semantic_result, f, ensure_ascii=False, indent=2)

print("event_type / severity asignados (simulando revisión manual):\n")
for g in semantic_result["groups"]:
    print(f"  Grupo #{g['group_id']}: event_type={g['event_type']!r}, severity={g['severity']!r}")
    print(f"    → {g['representative_template'][:80]}")

---
## PASO [4/4] — Generar Expresiones Regulares

In [ ]:
from template_generator.generate_regex_from_groups import process_groups_file, derive_output_path

process_groups_file(semantic_groups_path)

regex_path = derive_output_path(semantic_groups_path)
with open(regex_path, encoding="utf-8") as f:
    regex_data = json.load(f)

regex_groups = regex_data["groups"]
print(f"\n✓ {len(regex_groups)} grupo(s) con regex en: {regex_path}\n")
for g in regex_groups:
    print(f"GRUPO #{g['group_id']}  (event_type={g['event_type']}, severity={g['severity']})")
    print(f"  Template: {g['representative_template'][:80]}")
    print(f"  Regex:    {g['pattern'][:120]}{'...' if len(g['pattern']) > 120 else ''}")
    print()

In [ ]:
import re

compiled_by_group = {
    g["group_id"]: re.compile(g["pattern"], g["flags"]) for g in regex_groups
}

def clasificar_log_nuevo(mensaje, compiled_by_group, regex_groups):
    for g in regex_groups:
        if compiled_by_group[g["group_id"]].match(mensaje):
            return g["group_id"], g["event_type"], g["severity"]
    return None, None, None

logs_nuevos_reales = [
    "START DB Groovy",
    "Query que lanza: update proddta.f4211 set sdurab=1 where sdkcoo='01009' and sddcto='SO' and sddoco = '26020178' and sdlnid = '1000'",
    "Parameter Not Found: VICadena5",
    "Orchestration Exception Detected, exception saved in file: /u01/oracle/jde_home/LOGS_ORCH/5071516061834397254",
    "Something completely unrelated to any known pattern",
]

for log in logs_nuevos_reales:
    group_id, event_type, severity = clasificar_log_nuevo(log, compiled_by_group, regex_groups)
    print(f"LOG:        {log[:80]}")
    if group_id is not None:
        print(f"CLASIFICADO: grupo #{group_id}  |  event_type={event_type}  |  severity={severity}")
    else:
        print("CLASIFICADO: ❌ ningún grupo coincide")
    print("-" * 70)